# Module 3A — Predictive Maintenance Feature Engineering (Local)

This notebook prepares train-ready frames for Phase 3 models run **on your machine** (CPU is enough).

| Module | Dataset | Target |
|--------|---------|--------|
| 3B | `NEV_fault_dataset.csv` | `Fault Label` |
| 3C | `logistics_predictive_maintenanceV2.csv` | `Brake_Condition` (`Good` / `Fair` / `Poor`) |
| 3D | `EV_Battery_Dataset_1.csv` | `SOH_pct` |
| 3E | `logistics_predictive_maintenanceV2.csv` | `Tire_Wear_pct` (from `TPI`) |

CSVs must live under `data/predictive_maintenance/` (git-ignored). EVIoT SoH remains available for analysis but is excluded from 3D training because it is statistically disconnected from its telemetry.

In [ ]:
from pathlib import Path
import sys

# Repo root = parent of notebooks/
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT =", REPO_ROOT)

In [ ]:
from ml.predictive_maintenance import (
    DATA_DIR,
    dataset_inventory,
    datasets_ready,
    prepare_all_module_frames,
    summarize_frames,
)

print("DATA_DIR =", DATA_DIR)
print(dataset_inventory().to_string(index=False))
assert datasets_ready(), "Copy the four required CSVs into data/predictive_maintenance/ first"
print("All required datasets found.")

In [ ]:
frames = prepare_all_module_frames()
print(summarize_frames(frames))

## 3B — Engine / drivetrain fault (NEV)

In [ ]:
fault = frames["3B_engine_fault"]
display(fault.frame.head())
print("features:", fault.features)
print("target:", fault.target)
print(fault.y.value_counts().sort_index())
X_train, X_test, y_train, y_test = fault.train_test_split(stratify=True)
print("train/test:", len(X_train), len(X_test))

## 3C — Brake condition (logistics fleet)

In [ ]:
brake = frames["3C_brake_condition"]
display(brake.frame.describe())
print("features:", brake.features)
print("target classes:", brake.y.value_counts().sort_index().to_dict())
X_train, X_test, y_train, y_test = brake.train_test_split(stratify=True)
print("train/test:", len(X_train), len(X_test))

## 3D — Battery SoH (cycle aging, leakage-safe)

In [ ]:
battery = frames["3D_battery_soh"]
print(
    "Cycle SOH_pct rows:",
    len(battery.frame),
    "range:",
    float(battery.y.min()),
    "→",
    float(battery.y.max()),
)
print("features:", battery.features)
assert "Capacity_Ah" not in battery.features
display(battery.frame.head())

## 3E — Tire wear % (logistics TPI → Tire_Wear_pct)

In [ ]:
tire = frames["3E_tire_wear"]
cols = [c for c in ("Tire_Pressure", "TPI", "Tire_Wear_pct") if c in tire.frame.columns]
display(tire.frame[cols].describe())
print("features:", tire.features)
print("wear % range:", float(tire.y.min()), "→", float(tire.y.max()))
X_train, X_test, y_train, y_test = tire.train_test_split()
print("train/test:", len(X_train), len(X_test))

## Next steps

Train locally in dedicated notebooks (CPU-friendly):

1. **3B** — four-class NEV fault classifier
2. **3C** — XGBoost classifier on logistics `Brake_Condition`
3. **3D** — XGBoost on `SoH` / `SOH_pct`
4. **3E** — linear / tree model on `Tire_Wear_pct`

Save artifacts to `ml/models/*.joblib` as configured in `ml.predictive_maintenance.config`.